In [ ]:
#!pip install azure-functions-durable==1.2.9

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import sys
sys.path.append('../')
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [6]:
import azure.functions as func
import azure.durable_functions as df
import functools
from typing import Callable

from centraal_dataframework.blueprints import runner
from centraal_dataframework.resources import GreatExpectationsToolKit, get_context, get_datalake

AZURE_STORAGE_CONNECTION_STRING No configurado!


In [1]:
STR_FMT = 'TAREA: %(name)s--%(asctime)s-%(levelname)s-%(message)s'

In [5]:
bp = df.Blueprint()

## Internal working

1. first we need to define all the tasks as activity triggers, we will reuse the task decorator but add the durable decorator.

In [ ]:
def task(func: Callable):
    """Decorador para adicionar la tarea."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger = logging.getLogger(func.__name__)
        c_handler = logging.StreamHandler()
        fmt_log = logging.Formatter(STR_FMT)
        c_handler.setFormatter(fmt_log)
        logger.addHandler(c_handler)
        return func(get_datalake(), logger, *args, **kwargs)

    runner.add_task(wrapper, func.__name__)

    return wrapper

2. Then a pipeline will be an orchestator function, that calls the activity functions, We want create a way to define a pipeline very easy and internally add those tasks into the orchestation function.

This orchestation function will run everything and if any task fails, we will send an email, in this way we can group task into a `pipeline` in order to inform more that one `task` in the same email.


> note: this activity to send an email needs to be executed by another `task`, this task has to be general for all the `pipeline`. This is recomendation to comply with the recomendation: An orchestator should not impoklement logic to wait external systems of I/O tasks.



In [ ]:
@bp.route(route="startOrchestrator")
@bp.durable_client_input(client_name="client")
async def start_orchestrator(req: func.HttpRequest, client):
    instance_id = await client.start_new("my_orchestrator")
    
    logging.info(f"Started orchestration with ID = '{instance_id}'.")
    return client.create_check_status_response(req, instance_id)


3. We want to able to schedule these pipeline, so we wil create an instance of `runner` that will contains all those pipelines

> note: Is a table a proper a better way to storage this information?

The framework will have 3 `standard` functions:
1. a function that will have a timer trigger (60 minutes), that will check with the runner which pipelines needs to be executed

> note: we assume that this function will start each pipeline and with the control of [Configuration throttles](https://learn.microsoft.com/en-us/azure/azure-functions/durable/durable-functions-perf-and-scale#configuration-of-throttles) we will able to limit the concurrency, by default the framework enable one `pipeline` and one `task` each time. The important question is: Can we start all the pipelines needed and this configuration will manage the concurrency without any intervention?


also it seems nthat we need to take into account the `partitionCount`

2. a function that have an http trigger with a request containing the name of the pipeline to start

> note: the previuos question here also important, Can we start at demand a new `pipeline` and the durable fucntion framework will ad this `pipeline` to the end of the queue withoput nay intervention?

3. A function to get the status of a pipeline -> this can be useful to leverage other external monitoring acitivities. The idea is to have a http trigger.

> can we also use this to get the status of the pipeline running?